# Step 12: Other aggregation targets on QQQ and GLD

Companion to Lecture 3 of *Systematic Trading Notes*. Lecture 2 and notebook 11 covered one
family of aggregation: combining several bars into a smoothed estimate of price level. This
notebook covers five other, structurally different things the same causal-window idea can be
made to estimate instead — volatility, trend strength, flow, the relationship between two
instruments, and the shape of the recent return distribution — each tested against real data
rather than a constructed example. Where `trading_models` already has a tested implementation
(Wilder's ATR, ADX, the efficiency ratio), it is imported directly; where none exists (on-balance
volume, rolling correlation and beta), the computation is written locally in this notebook and
marked as such.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_models.data.ohlc import load_ohlc
from trading_models.momentum import average_true_range
from trading_models.plotting import SERIES, apply_theme, save_figure
from trading_models.trend_following.data.schema import InstrumentPanel
from trading_models.trend_following.signals.baselines.trend_strength import adx, efficiency_ratio

apply_theme()
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
FIG_DIR = ROOT / "figures" / "12_other_aggregation_targets"


## 1. Data

Two instruments: QQQ, the same series notebook 11 used, and GLD, chosen for Section 4
specifically because it is not a close substitute for QQQ — unlike a sector peer, its
correlation with an equity index is weak and regime-dependent, which is the more informative
case for a relationship statistic than two assets that are highly correlated almost by
construction.


In [2]:
START = "2015-01-01"


def load_with_volume(ticker: str, start: str) -> pd.DataFrame:
    ohlc = load_ohlc([ticker], start=start, cache=RAW / f"notebook12_{ticker.lower()}_ohlc.csv")
    out = pd.DataFrame({f: ohlc[f][ticker] for f in ("open", "high", "low", "close")})
    vol_cache = RAW / f"notebook12_{ticker.lower()}_volume.csv"
    if vol_cache.exists():
        out["volume"] = pd.read_csv(vol_cache, index_col=0, parse_dates=True)[ticker]
    else:
        import yfinance as yf

        raw = yf.download([ticker], start=start, auto_adjust=True, actions=False, progress=False)
        vol = raw["Volume"][ticker]
        vol.to_csv(vol_cache)
        out["volume"] = vol
    return out.dropna()


qqq = load_with_volume("QQQ", START)
gld = load_with_volume("GLD", START)
print(f"QQQ: {len(qqq)} bars, {qqq.index[0].date()} to {qqq.index[-1].date()}")
print(f"GLD: {len(gld)} bars, {gld.index[0].date()} to {gld.index[-1].date()}")

panel = InstrumentPanel(close=qqq[["close"]].rename(columns={"close": "QQQ"}),
                         high=qqq[["high"]].rename(columns={"high": "QQQ"}),
                         low=qqq[["low"]].rename(columns={"low": "QQQ"}))
EVENT = pd.Timestamp("2015-08-24")  # same event notebook 11 found as the widest-range QQQ bar


QQQ: 2960 bars, 2015-01-02 to 2026-10-09
GLD: 2960 bars, 2015-01-02 to 2026-10-09


## 2. Volatility: Wilder's Average True Range

$$
\mathrm{TR}_t = \max\big(H_t-L_t,\ \lvert H_t-C_{t-1}\rvert,\ \lvert L_t-C_{t-1}\rvert\big), \qquad
\mathrm{ATR}_t = \mathrm{ATR}_{t-1} + \tfrac{1}{\text{period}}\big(\mathrm{TR}_t-\mathrm{ATR}_{t-1}\big)
$$

Imported directly from `trading_models.momentum.average_true_range` — the same function the
project's trend-following signals already rely on for position sizing.


In [3]:
atr = average_true_range(qqq[["high"]].rename(columns={"high": "QQQ"}),
                          qqq[["low"]].rename(columns={"low": "QQQ"}),
                          qqq[["close"]].rename(columns={"close": "QQQ"}), period=14)["QQQ"]

zoom = qqq.loc[EVENT - pd.Timedelta(days=45): EVENT + pd.Timedelta(days=30)]
zi = zoom.index

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35],
                     vertical_spacing=0.06, subplot_titles=("QQQ", "ATR(14)"))
fig.add_candlestick(x=zi, open=zoom["open"], high=zoom["high"], low=zoom["low"], close=zoom["close"],
                     name="QQQ", showlegend=False, row=1, col=1)
fig.add_scatter(x=zi, y=atr.loc[zi], mode="lines", line=dict(width=1.8, color=SERIES[0]),
                 showlegend=False, row=2, col=1)
fig.update_layout(title=f"QQQ: ATR(14) around {EVENT.date()}", height=560, width=1000,
                   xaxis_rangeslider_visible=False, xaxis2_rangeslider_visible=False)
save_figure(fig, "12_01_atr", FIG_DIR)
fig


In [4]:
pre = atr.loc[:EVENT].iloc[-2]
peak = atr.loc[EVENT: EVENT + pd.Timedelta(days=10)].max()
peak_date = atr.loc[EVENT: EVENT + pd.Timedelta(days=10)].idxmax()
print(f"ATR(14) immediately before the event: {pre:.2f}")
print(f"ATR(14) peak: {peak:.2f} on {peak_date.date()} ({(peak / pre - 1):+.1%} vs pre-event)")


ATR(14) immediately before the event: 1.61
ATR(14) peak: 2.93 on 2015-08-26 (+81.6% vs pre-event)


## 3. Trend strength: efficiency ratio and ADX

Both answer "how trustworthy is the recent direction," not which direction it is, and both are
imported directly from `trading_models.trend_following.signals.baselines.trend_strength`.


In [5]:
er = efficiency_ratio(panel, window=20)["QQQ"]
adx20 = adx(panel, period=20)["QQQ"]

zoom_idx = er.loc[EVENT - pd.Timedelta(days=20): EVENT + pd.Timedelta(days=40)].index
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                     subplot_titles=("Efficiency ratio(20), unsigned", "ADX(20)"))
fig.add_scatter(x=zoom_idx, y=er.loc[zoom_idx], mode="lines", line=dict(width=1.8, color=SERIES[1]),
                 showlegend=False, row=1, col=1)
fig.add_scatter(x=zoom_idx, y=adx20.loc[zoom_idx], mode="lines", line=dict(width=1.8, color=SERIES[2]),
                 showlegend=False, row=2, col=1)
fig.update_yaxes(range=[0, 1], row=1, col=1)
fig.update_yaxes(range=[0, 100], row=2, col=1)
fig.update_layout(title=f"QQQ: trend-strength statistics around {EVENT.date()}", height=520, width=1000)
save_figure(fig, "12_02_trend_strength", FIG_DIR)
fig


In [6]:
print("Efficiency ratio(20) around the event:")
print(er.loc[EVENT - pd.Timedelta(days=3): EVENT + pd.Timedelta(days=10)].round(3))
print()
print("ADX(20) around the event:")
print(adx20.loc[EVENT - pd.Timedelta(days=3): EVENT + pd.Timedelta(days=10)].round(2))


Efficiency ratio(20) around the event:
Date
2015-08-21    0.442
2015-08-24    0.516
2015-08-25    0.590
2015-08-26    0.320
2015-08-27    0.224
2015-08-28    0.221
2015-08-31    0.247
2015-09-01    0.317
2015-09-02    0.241
2015-09-03    0.212
Name: QQQ, dtype: float64

ADX(20) around the event:
Date
2015-08-21    13.53
2015-08-24    16.41
2015-08-25    19.09
2015-08-26    21.51
2015-08-27    23.22
2015-08-28    24.77
2015-08-31    26.31
2015-09-01    27.99
2015-09-02    29.43
2015-09-03    30.54
Name: QQQ, dtype: float64


## 4. Flow: on-balance volume

$$
\mathrm{OBV}_t = \mathrm{OBV}_{t-1} + \begin{cases} +V_t & C_t>C_{t-1}\\ -V_t & C_t<C_{t-1}\\ 0 & C_t=C_{t-1}\end{cases}
$$

No equivalent exists in `trading_models`; computed locally. Unlike every statistic above, OBV
never decays — it is a plain running sum, not a rolling or an exponentially weighted one.


In [7]:
def obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    direction = np.sign(close.diff()).fillna(0.0)
    return (direction * volume).cumsum()


obv_qqq = obv(qqq["close"], qqq["volume"])

zoom_idx = obv_qqq.loc[EVENT - pd.Timedelta(days=45): EVENT + pd.Timedelta(days=30)].index
fig = go.Figure()
fig.add_scatter(x=zoom_idx, y=obv_qqq.loc[zoom_idx], mode="lines", line=dict(width=1.8, color=SERIES[3]))
fig.update_layout(title=f"QQQ: on-balance volume around {EVENT.date()}", height=420, width=1000, showlegend=False)
save_figure(fig, "12_03_obv", FIG_DIR)
fig


## 5. Relationship between series: QQQ and GLD

$$
\rho_t(N) = \mathrm{corr}\big(r^{QQQ}_{t-N+1:t},\, r^{GLD}_{t-N+1:t}\big), \qquad
\beta_t(N) = \frac{\mathrm{cov}\big(r^{QQQ}_{t-N+1:t},\, r^{GLD}_{t-N+1:t}\big)}{\mathrm{var}\big(r^{GLD}_{t-N+1:t}\big)}
$$

Neither exists in `trading_models` as a standalone utility; both are one-line `pandas` rolling
calls once the two return series are aligned.


In [8]:
both = pd.concat([qqq["close"].rename("QQQ"), gld["close"].rename("GLD")], axis=1).dropna()
ret = both.pct_change().dropna()

N = 20
rho = ret["QQQ"].rolling(N).corr(ret["GLD"])
cov = ret["QQQ"].rolling(N).cov(ret["GLD"])
beta = cov / ret["GLD"].rolling(N).var()

zoom_idx = rho.loc[EVENT - pd.Timedelta(days=60): EVENT + pd.Timedelta(days=60)].index
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                     subplot_titles=(f"Rolling correlation(QQQ, GLD), N={N}", f"Rolling beta(QQQ on GLD), N={N}"))
fig.add_scatter(x=zoom_idx, y=rho.loc[zoom_idx], mode="lines", line=dict(width=1.8, color=SERIES[0]),
                 showlegend=False, row=1, col=1)
fig.add_scatter(x=zoom_idx, y=beta.loc[zoom_idx], mode="lines", line=dict(width=1.8, color=SERIES[4]),
                 showlegend=False, row=2, col=1)
fig.add_hline(y=0, line=dict(color="#898781", width=1, dash="dot"), row=1, col=1)
fig.update_layout(title="QQQ vs. GLD: correlation and beta around the same event", height=520, width=1000)
save_figure(fig, "12_04_relationship", FIG_DIR)
fig


In [9]:
pre_window = rho.loc[:EVENT].iloc[-10:-5]
event_window = rho.loc[EVENT: EVENT + pd.Timedelta(days=10)]
print(f"Correlation(20), 5-10 bars before the event: {pre_window.mean():+.3f}")
print(f"Correlation(20), in the 10 bars after the event: min {event_window.min():+.3f}, max {event_window.max():+.3f}")
print(f"Beta(20), same before/after comparison: {beta.loc[pre_window.index].mean():+.3f} -> "
      f"min {beta.loc[event_window.index].min():+.3f}, max {beta.loc[event_window.index].max():+.3f}")


Correlation(20), 5-10 bars before the event: -0.119
Correlation(20), in the 10 bars after the event: min -0.363, max -0.146
Beta(20), same before/after comparison: -0.112 -> min -0.978, max -0.269


## 6. Distribution shape: rolling skewness and kurtosis of QQQ's own returns

`pandas`' built-in `.rolling(N).skew()` / `.kurt()` are used directly rather than hand-rolled:
they are well-tested and already implement the standard bias-adjusted sample estimators (not the
simple population-moment formulas used for the constructed example in Lecture 3, which converge
to the same values as $N$ grows but can differ visibly at small $N$).


In [10]:
ret_qqq = qqq["close"].pct_change()
skew20 = ret_qqq.rolling(N).skew()
kurt20 = ret_qqq.rolling(N).kurt()

zoom_idx = skew20.loc[EVENT - pd.Timedelta(days=30): EVENT + pd.Timedelta(days=60)].index
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                     subplot_titles=(f"Rolling skewness, N={N}", f"Rolling excess kurtosis, N={N}"))
fig.add_scatter(x=zoom_idx, y=skew20.loc[zoom_idx], mode="lines", line=dict(width=1.8, color=SERIES[2]),
                 showlegend=False, row=1, col=1)
fig.add_scatter(x=zoom_idx, y=kurt20.loc[zoom_idx], mode="lines", line=dict(width=1.8, color=SERIES[3]),
                 showlegend=False, row=2, col=1)
fig.add_hline(y=0, line=dict(color="#898781", width=1, dash="dot"), row=1, col=1)
fig.add_hline(y=0, line=dict(color="#898781", width=1, dash="dot"), row=2, col=1)
fig.update_layout(title="QQQ: rolling return-distribution shape around the event", height=520, width=1000)
save_figure(fig, "12_05_distribution_shape", FIG_DIR)
fig


In [11]:
print(f"Skewness(20) peak (most negative) within 15 bars of the event: "
      f"{skew20.loc[EVENT: EVENT + pd.Timedelta(days=22)].min():+.3f}")
print(f"Excess kurtosis(20) peak within 15 bars of the event: "
      f"{kurt20.loc[EVENT: EVENT + pd.Timedelta(days=22)].max():+.3f}")
calm_ref = kurt20.loc[EVENT - pd.Timedelta(days=120): EVENT - pd.Timedelta(days=30)]
print(f"Excess kurtosis(20), typical calm-period level (May-Jul 2015, n={calm_ref.notna().sum()}): "
      f"{calm_ref.median():+.3f}")


Skewness(20) peak (most negative) within 15 bars of the event: -1.459
Excess kurtosis(20) peak within 15 bars of the event: +2.971
Excess kurtosis(20), typical calm-period level (May-Jul 2015, n=63): +0.228


## 7. Summary

| Target | Statistic | Source |
|---|---|---|
| Volatility | ATR(14) | `trading_models.momentum.average_true_range` (reused) |
| Trend strength | Efficiency ratio(20), ADX(20) | `trading_models...trend_strength` (reused) |
| Flow | OBV | Local to this notebook |
| Relationship | Rolling correlation(20), beta(20) | Local to this notebook |
| Distribution shape | Rolling skew(20), kurtosis(20) | `pandas` built-in |

Every number above is read from real QQQ (and, for Section 5, GLD) prices around the same
real event notebook 11 used, not a constructed series — the comparison with Lecture 3's hand-built
numbers is between a controlled illustration and what the same formulas do on an actual market
day.
